# 069 — Modelos visión-lenguaje

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**CLIP (2021):** dos codificadores — imagen `f` y texto `g` — proyectan a un espacio común
con embeddings normalizados; la similitud coseno es un producto punto. Se entrena con
**aprendizaje contrastivo** (InfoNCE simétrica) sobre lotes de N pares imagen-texto de la
web (400 M): la matriz N×N de similitudes debe tener la diagonal alta. La **temperatura τ**
escala los logits: no cambia el ranking, cambia la confianza aparente.

**Zero-shot:** cada clase se convierte en un prompt ("una foto de un {perro}"), se codifica
con `g`, y la imagen se asigna a la clase de mayor coseno. Zero-shot ≠ sin datos: significa
sin *fine-tuning* por tarea.

**Límites:** embedding global (sabe *qué*, no *dónde* → sin grounding fino), comportamiento
de bolsa de palabras ("perro persigue gato" ≈ "gato persigue perro"), débil en conteo,
vulnerable a **ataques tipográficos** (texto impreso dentro de la imagen). Los VLM
generativos (Flamingo, LLaVA) conectan un encoder visual a un LLM para VQA y descripción
libre — heredando sus alucinaciones.


### 🧮 Cálculo de referencia (para los ejercicios)

```text
img = (0.8, 0.6, 0)   t_perro = (1,0,0)   t_gato = (0,1,0)   t_avión = (0,0,1)
cosenos: 0.8, 0.6, 0.0
softmax con τ=0.5 → logits (1.6, 1.2, 0) → p = (0.53, 0.36, 0.11) → "perro"
```


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("attention", seed=69)
show(result)


## Reflexión

1. Tu clasificador zero-shot funciona con fotos de catálogo pero falla con fotos tomadas
   por usuarios. ¿Qué diferencia de distribución lo explica y qué harías **sin** reentrenar
   CLIP (prompts, preprocesado, umbrales)?
2. Un producto trae impresa la palabra "banana" y CLIP lo clasifica como banana. ¿Cómo se
   llama este fallo, por qué ocurre en un espacio compartido imagen-texto y qué mitigación
   propondrías?
3. Si la clase verdadera no está en tu lista de prompts, ¿qué hace el argmax igualmente?
   ¿Cómo diseñarías una opción "ninguna de las anteriores" con umbrales de similitud?
